In [1]:
import pandas as pd
import glob
import os

output_file = "./data/clean_aqi_2020_2024.csv"

# 如果文件已存在就删除，避免重复写入
if os.path.exists(output_file):
    os.remove(output_file)

all_files = glob.glob(r"D:\Downloads\FYP CHINA\data\百度\*_*/*/*.csv", recursive=True)

for file in all_files:
    print(f"📥 处理文件: {file}")
    try:
        # 读取文件，只保留必要列
        df = pd.read_csv(file, encoding="utf-8")

        non_city_cols = ["date", "hour", "type", "Unnamed: 0"]
        city_cols = [col for col in df.columns if col not in non_city_cols]

        # 只处理重要污染物类型（减少内存）
        df = df[df["type"].isin(["AQI", "PM2.5", "PM10", "SO2", "NO2", "CO", "O3"])]

        # 转长表
        df_long = df.melt(id_vars=["date", "hour", "type"], value_vars=city_cols,
                          var_name="city", value_name="value")

        # 转宽表
        df_pivot = df_long.pivot_table(
            index=["date", "hour", "city"],
            columns="type",
            values="value",
            aggfunc='mean'   # 防止重复数据
        ).reset_index()

        df_pivot.columns.name = None

        # 转换日期
        df_pivot["date"] = pd.to_datetime(df_pivot["date"], format="%Y%m%d", errors="coerce")

        # 写入文件（逐步 append）
        df_pivot.to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file), encoding="utf-8")

        print(f"✔ 处理成功: {file}")

    except Exception as e:
        print(f"❌ 处理失败: {file}, 错误: {e}")

print("\n🎉 完成！数据已保存到：", output_file)


📥 处理文件: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200101.csv
✔ 处理成功: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200101.csv
📥 处理文件: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200102.csv
✔ 处理成功: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200102.csv
📥 处理文件: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200103.csv
✔ 处理成功: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200103.csv
📥 处理文件: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200104.csv
✔ 处理成功: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200104.csv
📥 处理文件: D:\Downloads\FYP CHINA\data\百度\城市_20200101-20201231\城市_20200101-20201231\china_cities_20200105.csv
✔ 处理成功: D:\Downloads\FYP CHINA\data\百

In [2]:
import pandas as pd

df = pd.read_csv("./data/clean_aqi_2020_2024.csv", encoding="utf-8")

df.head(5)
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98515780 entries, 0 to 98515779
Data columns (total 10 columns):
 #   Column  Dtype  
---  ------  -----  
 0   date    object 
 1   hour    int64  
 2   city    object 
 3   AQI     float64
 4   CO      float64
 5   NO2     float64
 6   O3      float64
 7   PM10    float64
 8   PM2.5   float64
 9   SO2     float64
dtypes: float64(7), int64(1), object(2)
memory usage: 7.3+ GB


,hour,AQI,CO,NO2,O3,PM10,PM2.5,SO2
count,9.851578e+07,9.545182e+07,9.744493e+07,9.762193e+07,9.737487e+07,9.607992e+07,9.720922e+07,9.775506e+07
mean,1.150405e+01,5.681929e+01,6.634498e-01,2.311004e+01,6.557759e+01,5.976619e+01,3.205331e+01,8.687394e+00
std,6.926604e+00,4.518053e+01,3.852741e-01,1.845965e+01,4.192423e+01,7.904858e+01,4.045115e+01,8.665234e+00
min,0.000000e+00,0.000000e+00,1.000000e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
25%,5.000000e+00,3.000000e+01,4.000000e-01,1.000000e+01,3.400000e+01,2.600000e+01,1.300000e+01,5.000000e+00
50%,1.200000e+01,4.700000e+01,6.000000e-01,1.800000e+01,6.000000e+01,4.400000e+01,2.300000e+01,7.000000e+00
75%,1.800000e+01,6.800000e+01,8.000000e-01,3.000000e+01,9.000000e+01,7.300000e+01,4.000000e+01,1.000000e+01
max,2.300000e+01,5.000000e+02,9.000000e+01,1.028000e+03,1.200000e+03,1.000000e+05,2.000000e+05,1.575000e+03


In [6]:
import pandas as pd
import os

df_all = pd.read_csv("./data/clean_aqi_2020_2024.csv", encoding="utf-8")

cities = df_all["city"].unique()

# 创建输出文件夹（重要！！）
os.makedirs("data/cities", exist_ok=True)

for city in cities:
    df_city = df_all[df_all["city"] == city]

    output_path = f"data/cities/{city}.csv"

    header = not os.path.exists(output_path)
    df_city.to_csv(output_path, mode="a", index=False, header=header)

print("🎉 拆分完成！每个城市已经有单独的 CSV 文件。")


KeyboardInterrupt: 

In [3]:
import os
import pandas as pd

# 你的城市文件夹路径
folder = r"D:\Downloads\FYP CHINA\data\cities"

# 读取所有文件名
files = os.listdir(folder)

# 只保留 .csv 文件（或你实际的扩展名）
city_files = [f for f in files if f.lower().endswith(".csv")]

# 提取城市名称（去掉 .csv）
cities = [os.path.splitext(f)[0] for f in city_files]

print("总城市文件数 =", len(cities))
print(cities)

# 检查中文文件名（排除英文与数字）
non_chinese = [c for c in cities if not all('\u4e00' <= ch <= '\u9fff' for ch in c)]

if non_chinese:
    print("\n⚠️ 存在非中文名称文件：")
    for c in non_chinese:
        print(" -", c)
else:
    print("\n所有文件名都是中文城市名 ✓")

# 保存城市列表到 CSV（可选）
df = pd.DataFrame({"city": cities})
df.to_csv("cities_list.csv", index=False, encoding="utf-8-sig")
print("\n城市列表已另存为 cities_list.csv")


总城市文件数 = 391
['七台河', '三亚', '三明', '三沙', '三门峡', '上海', '上饶', '东莞', '东营', '中卫', '中山', '临夏回族自治州', '临夏州', '临安', '临汾', '临沂', '临沧', '丹东', '丽水', '丽江', '义乌', '乌兰察布', '乌海', '乌鲁木齐', '乐山', '九江', '乳山', '云浮', '五家渠', '亳州', '伊春', '伊犁哈萨克州', '佛山', '佳木斯', '保定', '保山', '信阳', '儋州', '克孜勒苏柯尔克孜自治州', '克州', '克拉玛依', '六安', '六盘水', '兰州', '兰州新区', '兴安盟', '内江', '凉山州', '凉山彝族自治州', '包头', '北京', '北海', '十堰', '南京', '南充', '南宁', '南平', '南昌', '南通', '南阳', '博尔塔拉蒙古自治州', '博州', '即墨', '厦门', '双鸭山', '句容', '台州', '合肥', '吉安', '吉林', '吐鲁番地区', '吕梁', '吴忠', '吴江', '周口', '呼伦贝尔', '呼和浩特', '和田地区', '咸宁', '咸阳', '哈密地区', '哈尔滨', '唐山', '商丘', '商洛', '喀什地区', '嘉兴', '嘉峪关', '四平', '固原', '塔城地区', '大兴安岭地区', '大同', '大庆', '大理州', '大连', '天水', '天津', '太仓', '太原', '威海', '娄底', '孝感', '宁德', '宁波', '安庆', '安康', '安阳', '安顺', '定西', '宜兴', '宜宾', '宜昌', '宜春', '宝鸡', '宣城', '宿州', '宿迁', '富阳', '寿光', '山南', '岳阳', '崇左', '巴中', '巴彦淖尔', '巴音郭楞州', '常州', '常德', '常熟', '平凉', '平度', '平顶山', '广元', '广安', '广州', '庆阳', '库尔勒', '廊坊', '延安', '延边州', '延边朝鲜族自治州', '开封', '张家口', '张家港', '张家界', '张掖', '徐州', '德宏州', '德州', '德阳',

In [4]:
# data_clean_pipeline.py
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import chardet
import warnings
warnings.filterwarnings("ignore")

# --------------- 配置区（可按需修改） ----------------
SRC_FOLDER = r"D:\Downloads\FYP CHINA\data\cities"
OUT_FOLDER = r"D:\Downloads\FYP CHINA\data\cleaned_cities"
MASTER_OUT = r"D:\Downloads\FYP CHINA\data\cleaned_all_cities.csv"
SUMMARY_OUT = r"D:\Downloads\FYP CHINA\data\clean_summary.csv"

# 插值策略与阈值
INTERP_METHOD = "linear"
MAX_GAP_HOURS = 168   # 若连续缺失超过多少小时仍不插补（可视需要调整）
# 合理物理阈值（超出设为 NaN）
THRESHOLDS = {
    "AQI": (0, 1000),
    "PM2.5": (0, 5000),
    "PM10": (0, 10000),
    "NO2": (0, 5000),
    "SO2": (0, 5000),
    "O3": (0, 5000),
    "CO": (0, 100)   # 单位 mg/m3 可能不一样，谨慎
}
# -----------------------------------------------------

os.makedirs(OUT_FOLDER, exist_ok=True)
# 如果 master 已存在，先删除（重建）
if os.path.exists(MASTER_OUT):
    os.remove(MASTER_OUT)

def detect_encoding(file_path, n_bytes=20000):
    # 试探性探测编码
    with open(file_path, "rb") as f:
        raw = f.read(n_bytes)
    res = chardet.detect(raw)
    return res["encoding"] if res["encoding"] else "utf-8"

def unify_columns(df):
    # 将常见变体统一到目标字段名
    col_map = {}
    for c in df.columns:
        c0 = str(c).strip().lower()
        if c0 in ["date", "日期", "day"]:
            col_map[c] = "date"
        elif c0 in ["hour","小时","时"]:
            col_map[c] = "hour"
        elif "aqi" in c0:
            col_map[c] = "AQI"
        elif c0.replace(".","").replace("-","").replace("_","") in ["pm25","pm2.5","pm_25","pm2_5"]:
            col_map[c] = "PM2.5"
        elif "pm10" in c0:
            col_map[c] = "PM10"
        elif "so2" in c0:
            col_map[c] = "SO2"
        elif "no2" in c0:
            col_map[c] = "NO2"
        elif "o3" in c0:
            col_map[c] = "O3"
        elif "co" == c0 or c0.startswith("co "):
            col_map[c] = "CO"
        else:
            # 其余保留原名
            col_map[c] = c
    df = df.rename(columns=col_map)
    return df

def numeric_safe(series):
    return pd.to_numeric(series, errors="coerce")

def apply_thresholds(df):
    for col, (low, high) in THRESHOLDS.items():
        if col in df.columns:
            df.loc[(df[col] < low) | (df[col] > high), col] = np.nan
    return df

def process_city_file(file_path, city_name):
    # 1. 自动检测编码并读取
    try:
        enc = detect_encoding(file_path)
        df = pd.read_csv(file_path, encoding=enc)
    except Exception as e:
        # fallback
        df = pd.read_csv(file_path, encoding="utf-8", errors="replace")
    # 2. 统一列名
    df = unify_columns(df)
    # 3. ensure required columns exist
    # Some files may already be in cleaned format; handle common cases
    # If file already contains 'datetime' and pollutants, use it
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    else:
        # require date & hour columns
        if "date" not in df.columns:
            # try to infer from filename (if date embedded) or return empty
            df["date"] = np.nan
        if "hour" not in df.columns:
            # maybe it's daily file; assume hour=0
            df["hour"] = 0
        # normalize date strings (try various parse)
        df["date"] = df["date"].astype(str).str.strip()
        # convert date-like numeric YYYYMMDD -> YYYY-MM-DD
        df.loc[df["date"].str.match(r"^\d{8}$", na=False), "date"] = \
            pd.to_datetime(df.loc[df["date"].str.match(r"^\d{8}$", na=False), "date"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
        # parse to datetime
        df["hour"] = df["hour"].fillna(0).astype(int)
        # combine
        df["datetime"] = pd.to_datetime(df["date"], errors="coerce") + pd.to_timedelta(df["hour"], unit="h")
    # Drop rows without datetime
    df = df.dropna(subset=["datetime"])
    # 4. keep only columns we care about
    keep_cols = ["datetime", "AQI", "PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
    # convert numeric where present
    for c in keep_cols:
        if c in df.columns:
            df[c] = numeric_safe(df[c])
        else:
            df[c] = np.nan
    # 5. groupby datetime (in case duplicates) and aggregate mean
    df = df.groupby("datetime", as_index=False)[keep_cols[1:]].mean()
    # 6. reindex to full hourly range
    start = df["datetime"].min()
    end = df["datetime"].max()
    if pd.isna(start) or pd.isna(end):
        return None, {"city": city_name, "rows_before":0, "rows_after":0, "missing_after":0}
    full_idx = pd.date_range(start=start.floor('H'), end=end.ceil('H'), freq="H")
    df = df.set_index(pd.DatetimeIndex(df["datetime"])).reindex(full_idx)
    df.index.name = "datetime"
    df = df.drop(columns=["datetime"], errors="ignore")
    # 7. mark city column
    df["city"] = city_name
    # 8. apply thresholds (outliers -> NaN)
    df = apply_thresholds(df.reset_index())
    # 9. count missing before interp
    missing_before = df[["AQI","PM2.5","PM10","SO2","NO2","CO","O3"]].isna().sum().to_dict()
    # 10. Interpolate per column with limit for large gaps
    # use time interpolation (index is datetime)
    df = df.set_index("datetime")
    # limit: do not fill gaps larger than MAX_GAP_HOURS
    # create a helper mask to avoid interpolating over very long gaps
    # We'll do simple linear interpolation with limit = MAX_GAP_HOURS
    df_interp = df.copy()
    for col in ["AQI","PM2.5","PM10","SO2","NO2","CO","O3"]:
        if col in df_interp.columns:
            df_interp[col] = df_interp[col].interpolate(method="time", limit=MAX_GAP_HOURS, limit_direction="both")
    # 11. After interp, forward/backward fill small edges
    df_interp = df_interp.fillna(method="ffill").fillna(method="bfill")
    # 12. Round numeric columns reasonably
    for col in ["AQI","PM2.5","PM10","SO2","NO2","CO","O3"]:
        if col in df_interp.columns:
            df_interp[col] = df_interp[col].round(3)
    # 13. statistics
    rows_before = len(df)
    rows_after = len(df_interp)
    missing_after = df_interp.isna().sum().to_dict()
    # 14. reset index
    df_out = df_interp.reset_index()
    # 15. reorder columns
    cols_order = ["datetime","city","AQI","PM2.5","PM10","SO2","NO2","CO","O3"]
    df_out = df_out[cols_order]
    return df_out, {"city": city_name, "rows_before":rows_before, "rows_after":rows_after, "missing_before":missing_before, "missing_after":missing_after}

# ---------------- 批处理主流程 ----------------
city_files = [f for f in os.listdir(SRC_FOLDER) if f.lower().endswith(".csv")]
summary_list = []

for fname in tqdm(city_files, desc="Cities"):
    city = os.path.splitext(fname)[0]
    fpath = os.path.join(SRC_FOLDER, fname)
    try:
        df_city_clean, stats = process_city_file(fpath, city)
        if df_city_clean is None:
            print(f"⚠️ {city}: 处理后为空，跳过")
            continue
        # 保存 per-city clean file
        out_path = os.path.join(OUT_FOLDER, f"{city}.csv")
        df_city_clean.to_csv(out_path, index=False, encoding="utf-8")
        # append to master CSV (streaming)
        header = not os.path.exists(MASTER_OUT)
        df_city_clean.to_csv(MASTER_OUT, mode="a", index=False, header=header, encoding="utf-8")
        # add to summary
        stats_row = {
            "city": city,
            "rows_out": stats["rows_after"],
            "missing_after_AQI": stats["missing_after"].get("AQI", np.nan),
            "missing_after_PM2.5": stats["missing_after"].get("PM2.5", np.nan),
            "missing_after_PM10": stats["missing_after"].get("PM10", np.nan),
        }
        summary_list.append(stats_row)
    except Exception as e:
        print(f"❌ 处理失败: {city} -> {e}")

# 保存summary
df_summary = pd.DataFrame(summary_list)
df_summary.to_csv(SUMMARY_OUT, index=False, encoding="utf-8-sig")
print("全部完成。cleaned files at:", OUT_FOLDER)
print("master file:", MASTER_OUT)
print("summary:", SUMMARY_OUT)


Cities:   1%|          | 2/391 [00:00<00:50,  7.65it/s]

❌ 处理失败: 七台河 -> 'numpy.int64' object has no attribute 'floor'
❌ 处理失败: 三亚 -> 'numpy.int64' object has no attribute 'floor'


Cities:   1%|▏         | 5/391 [00:00<00:39,  9.89it/s]

❌ 处理失败: 三明 -> 'numpy.int64' object has no attribute 'floor'
❌ 处理失败: 三沙 -> 'numpy.int64' object has no attribute 'floor'
❌ 处理失败: 三门峡 -> 'numpy.int64' object has no attribute 'floor'


Cities:   2%|▏         | 7/391 [00:00<00:45,  8.44it/s]

...

❌ 处理失败: 黔南州 -> 'numpy.int64' object has no attribute 'floor'
❌ 处理失败: 黔南布依族苗族自治州 -> 'numpy.int64' object has no attribute 'floor'
❌ 处理失败: 黔西南州 -> 'numpy.int64' object has no attribute 'floor'


Cities: 100%|█████████▉| 390/391 [00:47<00:00,  8.95it/s]

❌ 处理失败: 黔西南布依族苗族自治州 -> 'numpy.int64' object has no attribute 'floor'
❌ 处理失败: 齐齐哈尔 -> 'numpy.int64' object has no attribute 'floor'


Cities: 100%|██████████| 391/391 [00:48<00:00,  8.14it/s]

❌ 处理失败: 龙岩 -> 'numpy.int64' object has no attribute 'floor'
全部完成。cleaned files at: D:\Downloads\FYP CHINA\data\cleaned_cities
master file: D:\Downloads\FYP CHINA\data\cleaned_all_cities.csv
summary: D:\Downloads\FYP CHINA\data\clean_summary.csv


In [5]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

input_path = r"D:\Downloads\FYP CHINA\data\cities"
output_path = r"D:\Downloads\FYP CHINA\data\cleaned_cities"

os.makedirs(output_path, exist_ok=True)

city_files = glob.glob(os.path.join(input_path, "*.csv"))

for file in tqdm(city_files, desc="Cities"):

    city = os.path.basename(file).replace(".csv", "")

    try:
        df = pd.read_csv(file, encoding="utf-8")

        # 删除重复
        df = df.drop_duplicates()

        # 转换日期
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

        # 空值处理：污染物为空 → 填为 NaN，不做 floor
        pollutants = ["AQI","PM2.5","PM10","SO2","NO2","CO","O3"]
        for col in pollutants:
            if col in df.columns:
                # 转成 float，再 floor，再转 int，避免报错
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[col] = np.floor(df[col].astype(float))

        # 保存文件
        df.to_csv(
            os.path.join(output_path, f"{city}.csv"),
            index=False,
            encoding="utf-8"
        )

    except Exception as e:
        print(f"❌ 处理失败: {city} -> {e}")


Cities: 100%|██████████| 391/391 [01:42<00:00,  3.82it/s]


In [6]:
import os
import pandas as pd
from tqdm import tqdm

root = r"D:\Downloads\FYP CHINA\data"
input_dir = os.path.join(root, "cleaned_cities")
output_dir = os.path.join(root, "daily_cities")

os.makedirs(output_dir, exist_ok=True)

# 自动识别污染物列（你不需要手动列）
def detect_pollutant_columns(df):
    exclude = {"city", "date", "hour"}
    return [col for col in df.columns if col not in exclude]

files = [f for f in os.listdir(input_dir) if f.endswith(".csv")]

print("Daily 生成开始，共有城市：", len(files))

for file in tqdm(files):
    path = os.path.join(input_dir, file)
    df = pd.read_csv(path, encoding="utf-8")

    # 日期格式标准化
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # 找出所有污染物列
    pollutant_cols = detect_pollutant_columns(df)

    # 按天聚合（平均值）
    daily = (
        df.groupby("date")[pollutant_cols]
        .mean(numeric_only=True)
        .reset_index()
    )

    # 加回 city 字段
    city_name = file.replace(".csv", "")
    daily.insert(0, "city", city_name)

    # 保存
    daily_file = os.path.join(output_dir, f"{city_name}_daily.csv")
    daily.to_csv(daily_file, index=False, encoding="utf-8")

print("\n全部城市 Daily 数据生成完毕！")
print("输出位置：", output_dir)


Daily 生成开始，共有城市： 391


100%|██████████| 391/391 [00:24<00:00, 16.22it/s]


全部城市 Daily 数据生成完毕！
输出位置： D:\Downloads\FYP CHINA\data\daily_cities


In [7]:
#read csv 
import pandas as pd
df = pd.read_csv("./data/daily_cities/北京_daily.csv", encoding="utf-8")
df.head(5)

,city,date,AQI,CO,NO2,O3,PM10,PM2.5,SO2
0,北京,2020-01-01,51.958333,0.041667,48.875000,10.291667,55.000000,33.708333,6.708333
1,北京,2020-01-02,70.375000,0.958333,62.541667,8.875000,78.916667,50.875000,7.333333
2,北京,2020-01-03,70.250000,0.708333,63.666667,13.875000,72.041667,50.500000,7.916667
3,北京,2020-01-04,60.750000,0.625000,58.583333,20.500000,66.000000,42.166667,7.125000
4,北京,2020-01-05,86.041667,1.000000,65.833333,7.208333,79.166667,63.708333,7.166667


In [1]:
import pandas as pd
import os
from tqdm import tqdm

CLEANED_DIR = r"D:\Downloads\FYP CHINA\data\cleaned_cities"
DAILY_DIR = r"D:\Downloads\FYP CHINA\data\daily_cities"
OUTPUT = r"D:\Downloads\FYP CHINA\data\master_daily.csv"

def load_folder(path):
    dfs = []
    files = [f for f in os.listdir(path) if f.endswith(".csv")]
    for f in tqdm(files, desc=f"Loading {path}"):
        fp = os.path.join(path, f)
        try:
            df = pd.read_csv(fp)
            # 标准化字段
            if "date" in df.columns:
                df["date"] = pd.to_datetime(df["date"], errors="coerce")
            dfs.append(df)
        except Exception as e:
            print("❌ Error:", fp, e)
    return dfs


print("📦 Loading cleaned cities...")
cleaned = load_folder(CLEANED_DIR)

print("📦 Loading daily cities...")
daily = load_folder(DAILY_DIR)

print("🔗 Merging...")
all_df = cleaned + daily

df = pd.concat(all_df, ignore_index=True)

print("🧹 Cleaning & Sorting...")
df = df.drop_duplicates()
df = df.dropna(subset=["city", "date"])
df = df.sort_values(["city", "date", "hour"])

print("💾 Saving to master_daily.csv...")
df.to_csv(OUTPUT, index=False, encoding="utf-8-sig")

print("🎉 Done!")
print("⚡ master_daily.csv 已生成，共记录:", len(df))


📦 Loading cleaned cities...


Loading D:\Downloads\FYP CHINA\data\cleaned_cities: 100%|██████████| 391/391 [00:22<00:00, 17.45it/s]


📦 Loading daily cities...


Loading D:\Downloads\FYP CHINA\data\daily_cities: 100%|██████████| 391/391 [00:09<00:00, 43.08it/s]


🔗 Merging...
🧹 Cleaning & Sorting...
💾 Saving to master_daily.csv...
🎉 Done!
⚡ master_daily.csv 已生成，共记录: 18066626


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("./data/master_daily.csv")
df["date"] = pd.to_datetime(df["date"])

df.head(5)

,date,hour,city,AQI,CO,NO2,O3,PM10,PM2.5,SO2
0,2020-01-01,0.0,七台河,63.0,0.0,44.0,29.0,58.0,45.0,17.0
1,2020-01-01,1.0,七台河,70.0,0.0,39.0,33.0,56.0,51.0,21.0
2,2020-01-01,2.0,七台河,74.0,0.0,47.0,27.0,63.0,54.0,18.0
3,2020-01-01,3.0,七台河,66.0,0.0,37.0,39.0,56.0,48.0,15.0
4,2020-01-01,4.0,七台河,60.0,0.0,34.0,36.0,58.0,43.0,17.0
